# 02_feature_engineering.ipynb

This notebook loads the raw dataset from Amazon S3, performs exploratory data profiling, applies feature engineering transformations, creates and populates a SageMaker Feature Store feature group, and prepares the processed dataset for downstream model training.

## Import Project Modules

The project source code is organized in the `src/` directory at the repository root.

Because this notebook is executed from the `notebooks/` directory, the project root is added to the Python path so that modules from the `src` package can be imported.

In [1]:
import sys

sys.path.append("..")

## Import Dependencies

Import the required Python libraries and project modules.

In [2]:
import time
from pathlib import Path

import boto3
import pandas as pd

from sklearn.model_selection import train_test_split

from src.config import (
    AWS_REGION,
    BUCKET_NAME,
    FEATURE_GROUP_NAME,
    FEATURE_STORE_S3_URI,
    PROCESSED_DATA_KEY,
    RAW_DATA_KEY,
    TRAIN_DATA_KEY,
    VALIDATION_DATA_KEY,
    TEST_DATA_KEY,
    TARGET_COLUMN,
)
from src.preprocess_simplified import preprocess_dataframe, split_data

## Initialize AWS Clients

Initialize clients for Amazon S3, SageMaker, SageMaker Feature Store and STS.,

In [3]:
s3 = boto3.client("s3", region_name=AWS_REGION)
sm = boto3.client("sagemaker", region_name=AWS_REGION)
featurestore_runtime = boto3.client("sagemaker-featurestore-runtime", region_name=AWS_REGION)
sts = boto3.client("sts", region_name=AWS_REGION)

## Load Raw Dataset from Amazon S3

The raw dataset created during the ingestion step is loaded from Amazon S3 for profiling and feature engineering.

In [4]:
obj = s3.get_object(
    Bucket=BUCKET_NAME,
    Key=RAW_DATA_KEY
)

df = pd.read_csv(obj["Body"])

## Dataset Preview

Inspect the first rows of the dataset and verify that the raw data was loaded correctly.

In [5]:
df.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,real estate,67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,real estate,22,none,own,1,skilled,1,none,yes,bad
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,good
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,...,life insurance,45,none,for free,1,skilled,2,none,yes,good
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,...,no known property,53,none,for free,2,skilled,2,none,yes,bad


## Data Quality Validation

Perform a quick validation to inspect missing values and confirm the dataset structure.

In [6]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

missing_values = df.isnull().sum()

print(f"Total missing values: {missing_values.sum()}")

Rows: 1000
Columns: 21
Total missing values: 0


## Data Profiling Summary

The data profiling step in AWS SageMaker Data Wrangler identified the following observations:

- No missing values were detected in the dataset.
- The target distribution is moderately imbalanced, with approximately 70% `good` and 30% `bad`.
- Numerical feature distributions show meaningful variation across the dataset.
- Several moderate correlations were identified between selected features.
- The dataset contains a mixture of numerical and categorical features.

## Data Processing

No missing values are detected during data profiling, and therefore no imputation is required.

Categorical features are transformed using one-hot encoding in SageMaker Data Wrangler to prepare the dataset for model training with XGBoost.

The transformation workflow is implemented in Data Wrangler and exported as Python code for reuse within the project and future pipeline automation.

In [7]:
processed_df = preprocess_dataframe(df)

## Preview Processed Dataset

Inspect the first rows of the processed dataset.

In [8]:
processed_df.head()

,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents,class,checking_status_0le_Xlt_200,checking_status_lt_0,...,job_skilled,job_unemp_unskilled_non_res,job_unskilled_resident,own_telephone_none,own_telephone_yes,foreign_worker_no,foreign_worker_yes,other_payment_plans_bank,other_payment_plans_none,other_payment_plans_stores
0,6,1169,4,4,67,2,1,1,0,1,...,1,0,0,0,1,0,1,0,1,0
1,48,5951,2,2,22,1,1,0,1,0,...,1,0,0,1,0,0,1,0,1,0
2,12,2096,2,3,49,1,2,1,0,0,...,0,0,1,1,0,0,1,0,1,0
3,42,7882,2,4,45,1,2,1,0,1,...,1,0,0,1,0,0,1,0,1,0
4,24,4870,3,4,53,2,2,0,0,1,...,1,0,0,1,0,0,1,0,1,0


## Prepare Feature Store Dataset

Prepare the processed dataset for SageMaker Feature Store by adding metadata columns and removing the target variable.

In [ ]:
# Start from the processed dataframe
feature_df = processed_df.drop(columns=[TARGET_COLUMN]).copy()

# Feature Store metadata columns
feature_df["record_id"] = feature_df.index.astype(str)
feature_df["event_time"] = pd.Timestamp.utcnow().strftime("%Y-%m-%dT%H:%M:%SZ")

feature_df.head()

## Retrieve SageMaker Execution Role

Retrieve the execution role associated with the SageMaker domain. This role is required to create and manage SageMaker Feature Store resources.

In [ ]:
domain = sm.list_domains()["Domains"][0]

domain_details = sm.describe_domain(
    DomainId=domain["DomainId"]
)

ROLE_ARN = domain_details["DefaultUserSettings"]["ExecutionRole"]
print(ROLE_ARN)

## Create Feature Definitions

Create the Feature Store schema by converting dataframe column types into SageMaker Feature Store feature definitions.

In [ ]:
feature_definitions = []

for column, dtype in feature_df.dtypes.items():

    if str(dtype) == "object":
        feature_type = "String"

    elif str(dtype) in ["int64", "int32", "bool"]:
        feature_type = "Integral"

    elif str(dtype) in ["float64", "float32"]:
        feature_type = "Fractional"

    else:
        raise ValueError(
            f"Unsupported dtype '{dtype}' for column '{column}'"
        )

    feature_definitions.append(
        {
            "FeatureName": column,
            "FeatureType": feature_type,
        }
    )
feature_definitions

## Create Feature Group

Create the SageMaker Feature Store feature group using the generated feature definitions.

In [ ]:
try:
    sm.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )

    print(f"Feature Group '{FEATURE_GROUP_NAME}' already exists.")

except sm.exceptions.ResourceNotFound:
    print(f"Creating Feature Group '{FEATURE_GROUP_NAME}'...")

    sm.create_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME,
        RecordIdentifierFeatureName="record_id",
        EventTimeFeatureName="event_time",
        FeatureDefinitions=feature_definitions,
        OnlineStoreConfig={
            "EnableOnlineStore": True
        },
        OfflineStoreConfig={
            "S3StorageConfig": {
                "S3Uri": FEATURE_STORE_S3_URI
            }
        },
        RoleArn=ROLE_ARN,
    )

## Wait for Feature Group Creation

Wait until the feature group status changes to `Created` before ingesting any records.

In [ ]:
while True:
    status = sm.describe_feature_group(
        FeatureGroupName=FEATURE_GROUP_NAME
    )["FeatureGroupStatus"]

    print(f"Status: {status}")

    if status == "Created":
        print("Feature Group is ready.")
        break

    if status == "CreateFailed":
        raise RuntimeError("Feature Group creation failed.")

    time.sleep(10)

## Ingest Features into Feature Store

Write the processed feature records to SageMaker Feature Store.

In [ ]:
for _, row in feature_df.iterrows():

    record = []

    for col in feature_df.columns:
        record.append(
            {
                "FeatureName": col,
                "ValueAsString": str(row[col])
            }
        )

    featurestore_runtime.put_record(
        FeatureGroupName=FEATURE_GROUP_NAME,
        Record=record
    )

## Verify Feature Store Records

Read back a sample record to confirm that ingestion worked as expected.

In [ ]:
featurestore_runtime.get_record(
    FeatureGroupName=FEATURE_GROUP_NAME,
    RecordIdentifierValueAsString="0"
)

## Split Dataset into Training, Validation and Test Sets

The processed dataset is split into training, validation, and test subsets.

The training set is used to fit the model, the validation set is used to evaluate model performance during development, and the test set is reserved for final model assessment.

Stratified sampling is applied to preserve the original class distribution across all subsets.

In [26]:
train_df, validation_df, test_df = split_data(
    processed_df,
    test_size=0.2,
    validation_size=0.2,
    random_state=42,
)

In [31]:
ls data 

clarify/  datasets/  evaluation/  models/  processed/  raw/  registry/


In [15]:
print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(validation_df)}")
print(f"Test samples: {len(test_df)}")

Training samples: 600
Validation samples: 200
Test samples: 200


## Upload Processed Data to S3

Store a local copy of the processed dataset and upload it to the processed data layer in Amazon S3.

In [16]:
Path("data/processed").mkdir(
    parents=True,
    exist_ok=True
)

In [17]:
processed_file = "data/processed/german_credit.csv"

processed_df.to_csv(
    processed_file,
    index=False
)

In [18]:
s3.upload_file(
    processed_file,
    BUCKET_NAME,
    PROCESSED_DATA_KEY
)

print(f"Uploaded processed data to s3://{BUCKET_NAME}/{PROCESSED_DATA_KEY}")

Uploaded processed data to s3://aws-sagemaker-showcase/processed/german_credit_processed.csv


In [19]:
s3.head_object(
    Bucket=BUCKET_NAME,
    Key=PROCESSED_DATA_KEY
)

print("Upload for processed file successful")

Upload for processed file successful


## Upload Split Datasets to Amazon S3

Store local copies of the training, validation, and test datasets and upload them to Amazon S3 for use in downstream training, evaluation, and MLOps workflows.

In [20]:
Path("data/datasets").mkdir(
    parents=True,
    exist_ok=True
)

In [21]:
train_file = "data/datasets/german_credit_train.csv"
validation_file = "data/datasets/german_credit_validation.csv"
test_file = "data/datasets/german_credit_test.csv"

train_df.to_csv(
    train_file,
    index=False
)

validation_df.to_csv(
    validation_file,
    index=False
)

test_df.to_csv(
    test_file,
    index=False
)

In [22]:
s3.upload_file(
    train_file,
    BUCKET_NAME,
    TRAIN_DATA_KEY
)

print(f"Uploaded train file to s3://{BUCKET_NAME}/{TRAIN_DATA_KEY}")

s3.upload_file(
    validation_file,
    BUCKET_NAME,
    VALIDATION_DATA_KEY
)

print(f"Uploaded validation file to s3://{BUCKET_NAME}/{VALIDATION_DATA_KEY}")

s3.upload_file(
    test_file,
    BUCKET_NAME,
    TEST_DATA_KEY
)

print(f"Uploaded test file to s3://{BUCKET_NAME}/{TEST_DATA_KEY}")

Uploaded train file to s3://aws-sagemaker-showcase/datasets/german_credit_train.csv
Uploaded validation file to s3://aws-sagemaker-showcase/datasets/german_credit_validation.csv
Uploaded test file to s3://aws-sagemaker-showcase/datasets/german_credit_test.csv


In [23]:
for key in [
    TRAIN_DATA_KEY,
    VALIDATION_DATA_KEY,
    TEST_DATA_KEY,
]:
    s3.head_object(
        Bucket=BUCKET_NAME,
        Key=key,
    )

print("All dataset uploads successful")

All dataset uploads successful


## Result

The processed dataset is now stored in Amazon S3.